In [1]:
import os
Root = r"C:\Users\Wasim\Desktop\Speech_emotin_Recognisation\dataset\speech-emotion-recognition-ravdess-data"
os.chdir(Root)

In [2]:
import librosa
import soundfile
import os, glob, pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

In [11]:
#Extract features (mfcc, chroma, mel) from a sound file
def extract_feature(file_name, mfcc, chroma, mel):
    with soundfile.SoundFile(file_name) as sound_file:
        X = sound_file.read(dtype="float32")
        sample_rate=sound_file.samplerate
        if chroma:
            stft=np.abs(librosa.stft(X))
        result=np.array([])
        if mfcc:
            mfccs=np.mean(librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=40).T, axis=0)
            result=np.hstack((result, mfccs))
        if chroma:
            chroma=np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate).T,axis=0)
            result=np.hstack((result, chroma))
        if mel:
            mel=np.mean(librosa.feature.melspectrogram(y=X, sr=sample_rate).T,axis=0)
            result=np.hstack((result, mel))
    return result

In [12]:
# Emotions in the RAVDESS dataset
emotions={
  '01':'neutral',
  '02':'calm',
  '03':'happy',
  '04':'sad',
  '05':'angry',
  '06':'fearful',
  '07':'disgust',
  '08':'surprised'
}

#Emotions to observe
observed_emotions=['calm', 'happy', 'fearful', 'disgust']

In [13]:
#Load the data and extract features for each sound file
def load_data(test_size=0.2):
    x,y=[],[]
    for file in glob.glob("C:\\Users\\Wasim\\Desktop\\Speech_emotin_Recognisation\\dataset\\speech-emotion-recognition-ravdess-data\\Actor_*\\*.wav"):
        file_name=os.path.basename(file)
        emotion=emotions[file_name.split("-")[2]]
        if emotion not in observed_emotions:
            continue
        feature=extract_feature(file, mfcc=True, chroma=True, mel=True)
        x.append(feature)
        y.append(emotion)
    return train_test_split(np.array(x), y, test_size=test_size, random_state=9)

In [14]:
#Split the dataset
x_train,x_test,y_train,y_test=load_data(test_size=0.25)

In [15]:
x_train

array([[-5.22061890e+02,  3.50668907e+01,  3.75342917e+00, ...,
         1.65243153e-04,  1.04321596e-04,  6.55571566e-05],
       [-6.41227722e+02,  4.49487762e+01, -1.85174072e+00, ...,
         3.89261913e-05,  3.05255344e-05,  2.94166621e-05],
       [-6.50705750e+02,  5.30211639e+01, -4.92040396e+00, ...,
         4.75216839e-05,  3.46632551e-05,  1.62844444e-05],
       ...,
       [-5.50096191e+02,  1.70297680e+01, -1.14575634e+01, ...,
         1.51764631e-04,  1.16828531e-04,  8.47479314e-05],
       [-5.55357605e+02,  4.71569710e+01,  1.10750732e+01, ...,
         1.61086457e-04,  1.04962477e-04,  6.52811723e-05],
       [-5.04816345e+02,  3.53618660e+01, -1.43495789e+01, ...,
         6.08151488e-04,  5.55269769e-04,  4.47782222e-04]],
      shape=(576, 180))

In [16]:
#Get the shape of the training and testing datasets
print((x_train.shape[0], x_test.shape[0]))

(576, 192)


In [17]:
#Get the number of features extracted
print(f'Features extracted: {x_train.shape[1]}')

Features extracted: 180


In [18]:
#Initialize the Multi Layer Perceptron Classifier
model=MLPClassifier(alpha=0.01, batch_size=256, epsilon=1e-08, hidden_layer_sizes=(300,), learning_rate='adaptive', max_iter=500)

In [19]:
#Train the model
model.fit(x_train,y_train)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(300,)"
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.01
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",256
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'adaptive'
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",500
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",None


In [20]:
#Predict for the test set
y_pred=model.predict(x_test)

In [21]:
y_pred

array(['happy', 'fearful', 'happy', 'happy', 'fearful', 'calm', 'calm',
       'happy', 'calm', 'fearful', 'happy', 'fearful', 'fearful', 'happy',
       'disgust', 'happy', 'calm', 'fearful', 'happy', 'calm', 'disgust',
       'disgust', 'disgust', 'calm', 'happy', 'happy', 'fearful', 'happy',
       'fearful', 'fearful', 'happy', 'fearful', 'happy', 'fearful',
       'happy', 'calm', 'calm', 'fearful', 'calm', 'disgust', 'happy',
       'calm', 'fearful', 'calm', 'fearful', 'disgust', 'disgust', 'calm',
       'calm', 'happy', 'fearful', 'fearful', 'fearful', 'fearful',
       'happy', 'fearful', 'calm', 'happy', 'calm', 'calm', 'disgust',
       'calm', 'happy', 'calm', 'disgust', 'calm', 'calm', 'disgust',
       'fearful', 'fearful', 'fearful', 'fearful', 'fearful', 'fearful',
       'fearful', 'disgust', 'fearful', 'happy', 'calm', 'fearful',
       'disgust', 'calm', 'fearful', 'calm', 'disgust', 'fearful',
       'disgust', 'fearful', 'happy', 'fearful', 'disgust', 'fearful',
 

In [22]:
#Calculate the accuracy of our model
accuracy=accuracy_score(y_true=y_test, y_pred=y_pred)

#Print the accuracy
print("Accuracy: {:.2f}%".format(accuracy*100))

Accuracy: 65.10%


In [23]:
from sklearn.metrics import accuracy_score, f1_score

In [24]:
f1_score(y_test, y_pred,average=None)

array([0.78181818, 0.575     , 0.59813084, 0.62068966])

In [25]:
import pandas as pd
df=pd.DataFrame({'Actual': y_test, 'Predicted':y_pred})
df.head(20)

,Actual,Predicted
0,happy,happy
1,calm,fearful
2,happy,happy
3,happy,happy
4,disgust,fearful
5,calm,calm
6,happy,calm
7,happy,happy
8,disgust,calm
9,happy,fearful


In [26]:
import pickle
# Writing different model files to file
with open( 'modelForPrediction1.sav', 'wb') as f:
    pickle.dump(model,f)

In [28]:
filename = 'modelForPrediction1.sav'
loaded_model = pickle.load(open(filename, 'rb')) # loading the model file from the storage

feature=extract_feature("C:\\Users\\Wasim\\Desktop\\Speech_emotin_Recognisation\\dataset\\speech-emotion-recognition-ravdess-data\\Actor_07\\03-01-07-01-01-01-07.wav", mfcc=True, chroma=True, mel=True)

feature=feature.reshape(1,-1)

prediction=loaded_model.predict(feature)
prediction

array(['disgust'], dtype='<U7')

In [29]:
feature

array([[-6.72173340e+02,  4.76361618e+01,  6.20053434e+00,
         1.50347681e+01, -5.50858068e+00, -8.99175549e+00,
        -6.98249626e+00, -9.17137206e-01, -4.93930817e+00,
        -1.27421331e+00, -2.68863034e+00, -1.73354816e+00,
        -1.58014476e+00,  9.12489772e-01, -7.82483482e+00,
        -1.08540738e+00, -4.21403646e+00, -1.02785611e+00,
        -7.53487587e+00, -1.62019670e+00, -5.31426620e+00,
        -5.34919071e+00, -1.65086222e+00, -1.26491487e+00,
        -3.24852324e+00, -1.17159700e+00, -4.45703745e+00,
        -3.71928740e+00, -1.79072833e+00, -2.60678053e+00,
        -1.98549461e+00, -2.51228833e+00, -4.47960758e+00,
        -2.34208536e+00, -1.66583693e+00, -3.54760498e-01,
        -2.14420319e+00, -3.92549729e+00, -2.74577355e+00,
        -1.04235947e+00,  5.96110284e-01,  6.32153094e-01,
         5.88532329e-01,  6.02480292e-01,  5.80018461e-01,
         5.56198657e-01,  5.86232424e-01,  5.95310271e-01,
         6.45055294e-01,  6.19390547e-01,  6.08830929e-0